# Text Fundamentals Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Text is data.** Chaining works because every string method returns a NEW string — `lower()` then `strip()` then `split()`.

In [ ]:
raw_review = "  LOVED it!!! Best purchase ever 😍  "

words = raw_review.lower().strip().split()
print(words)
# ['loved', 'it!!!', 'best', 'purchase', 'ever', '😍']

print(len(raw_review), "characters including spaces")   # 36 - emoji counts as ONE character
print(repr(raw_review))   # repr() reveals the hidden edge whitespace

**2. Glue and swap.** `join` glues a list into a string; methods never change the original, so assign or chain.

In [ ]:
ticket = "  ORDER #4521 has not arrived. Please HELP!  "

print("-".join(["urgent", "order", "4521"]))   # urgent-order-4521
print(ticket.strip().startswith("ORDER"))      # True - strip() first!
print(ticket.strip().replace("#", "No."))      # ORDER No.4521 has not arrived. Please HELP!
print(ticket.strip().split()[:4])              # ['ORDER', '#4521', 'has', 'not']

**3. Code points both ways.** A `str` is a sequence of Unicode code points; UTF-8 is the byte recipe — and the two lengths disagree.

In [ ]:
smiley = "🙂"

print(ord("A"), ord("a"))     # 65 97
print(chr(2535))              # ১ - Bengali digit one, any number works
print(len(smiley), "character,", len(smiley.encode("utf-8")), "bytes")
# 1 character, 4 bytes - len() counts CODE POINTS, encoding counts BYTES

## Part 2 — Practice

**4. The flag surprise.** Flags are not single symbols: they are TWO regional-indicator code points that fonts merge visually.

In [ ]:
flag = "🇧🇩"

print(flag, "has length:", len(flag))    # 2 - two regional indicators!
print([hex(ord(ch)) for ch in flag])     # ['0x1f1e7', '0x1f1e9']

print(flag[0], "<- slicing tears off a lone indicator")
# Slicing cuts CODE POINTS, so it can rip merged sequences apart.

**5. Two spellings of one word.** Same visible letters, different internal spelling: composed `U+00E9` vs `e` + combining accent. NFC collapses them.

In [ ]:
import unicodedata

e1 = "café"          # composed: é is ONE code point (U+00E9)
e2 = "cafe\u0301"    # decomposed: e + COMBINING ACUTE ACCENT

print(e1 == e2, "| lengths:", len(e1), "vs", len(e2))
# False | lengths: 4 vs 5 - to your eyes they match, to == they do not

nfc = unicodedata.normalize("NFC", e2)
print("after NFC equal?", nfc == e1, "| length:", len(nfc))
# True | length: 4 - normalize EVERYTHING once, early, always the same way

**6. Profile your corpus.** Flatten with a double comprehension, count with `len` and `set`; the ratio measures lexical richness.

In [ ]:
REVIEWS = [
    "The battery life is great and the screen is bright.",
    "Battery drains fast but the screen looks stunning.",
    "Great camera, poor battery, great price though.",
    "The speaker is loud but the battery is poor.",
    "Bright screen, great speakers, fair price. Loved it!",
    "Poor packaging but fast delivery and a great price.",
]

flat = [w.strip(".,!?") for doc in REVIEWS for w in doc.lower().split()]

total_words = len(flat)
vocab = set(flat)
ttr = len(vocab) / total_words

print("documents :", len(REVIEWS))
print("total words:", total_words)              # 51
print("vocabulary :", len(vocab))               # 26 unique words
print("type-token ratio:", round(ttr, 3))       # 0.51 - repeated words lower richness

**7. Rank the words.** `Counter(...).most_common(k)` ranks instantly; the champions are usually glue words like `the`.

In [ ]:
from collections import Counter

REVIEWS = [
    "The battery life is great and the screen is bright.",
    "Battery drains fast but the screen looks stunning.",
    "Great camera, poor battery, great price though.",
    "The speaker is loud but the battery is poor.",
    "Bright screen, great speakers, fair price. Loved it!",
    "Poor packaging but fast delivery and a great price.",
]
flat = [w.strip(".,!?") for doc in REVIEWS for w in doc.lower().split()]

for word, n in Counter(flat).most_common(6):
    print(f"{word:>10} | {n} {'#' * n}")
# the/great/battery/is lead - mostly stopword glue plus our topic words.
# That is exactly why TF-IDF (later lessons) downweights everywhere-words.

## Part 3 — Challenge

**8. Why the naive splitter fails.** One character class, three casualties: abbreviations, decimals, and the punctuation itself.

In [ ]:
import re

story = "Dr. Smith paid $3.50 for tea. Then he walked home!"

naive_parts = [p for p in re.split(r"[.!?]", story) if p]
for p in naive_parts:
    print(repr(p))
# ['Dr', ' Smith paid $3', '50 for tea', ' Then he walked home']
#
# Three failures at once:
# 1. the abbreviation Dr. lost its dot
# 2. the decimal $3.50 was shredded into $3 + 50
# 3. the sentence punctuation itself vanished

**9. Repair the splitter with lookbehind.** `(?<!...)` matches only when the preceding text does NOT look like that — fixed width per assertion, so one line per abbreviation.

In [ ]:
import re

sentence_re = re.compile(
    r"""
    (?<=[.!?])     # position sits right after . ! or ?
    (?<!Dr\.)      # ...but NOT right after these title abbreviations
    (?<!Mr\.)
    (?<!Mrs\.)
    (?<!Ms\.)
    (?<!St\.)
    \s+            # finally, split on the whitespace that follows
    """,
    re.VERBOSE,
)

story = "Dr. Smith paid $3.50 for tea. Then he walked home!"
for s in sentence_re.split(story):
    print(repr(s))
# ['Dr. Smith paid $3.50 for tea.', 'Then he walked home!'] - healed

harder = (
    "Mrs. Rahman ordered 2.5 kg of rice. It cost $12.75!! "
    "Was she happy? Yes. St. Martin tours were next."
)
for s in sentence_re.split(harder):
    print("-", s.strip())
# decimals survive ($12.75, 2.5), titles survive (Mrs., St.),
# and real sentences still separate